In [1]:
from tqdm.notebook import tqdm

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
# check if cuda is available
if torch.cuda.is_available():
    device = "cuda"
    torch.cuda.empty_cache()
else:
    device = "cpu"

### Tiny LLM built on Pan Tadeusz or Odyssey

In [ ]:
# get training text

In [30]:
!wget https://wolnelektury.pl/media/book/txt/orwell-rok-1984.txt

--2025-05-22 08:36:41--  https://wolnelektury.pl/media/book/txt/orwell-rok-1984.txt
Resolving wolnelektury.pl (wolnelektury.pl)... 51.83.143.148, 2001:41d0:602:3294::
Connecting to wolnelektury.pl (wolnelektury.pl)|51.83.143.148|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 612045 (598K) [text/plain]
Saving to: ‘orwell-rok-1984.txt’

orwell-rok-1984.txt 100%[===================>] 597.70K   659KB/s    in 0.9s    

2025-05-22 08:36:43 (659 KB/s) - ‘orwell-rok-1984.txt’ saved [612045/612045]



In [ ]:
#!wget https://classics.mit.edu/Homer/odyssey.mb.txt

In [63]:
!wget https://wolnelektury.pl/media/book/txt/lalka-tom-pierwszy.txt

--2025-05-22 08:57:48--  https://wolnelektury.pl/media/book/txt/lalka-tom-pierwszy.txt
Resolving wolnelektury.pl (wolnelektury.pl)... 51.83.143.148, 2001:41d0:602:3294::
Connecting to wolnelektury.pl (wolnelektury.pl)|51.83.143.148|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 860296 (840K) [text/plain]
Saving to: ‘lalka-tom-pierwszy.txt’

lalka-tom-pierwszy. 100%[===================>] 840.13K   931KB/s    in 0.9s    

2025-05-22 08:57:50 (931 KB/s) - ‘lalka-tom-pierwszy.txt’ saved [860296/860296]



In [31]:
lines = []
with open("orwell-rok-1984.txt") as f:
    for l in f:
        lines.append(l.strip())

# devide into stanzas
stanzas = []
i_prev = 0
for i in range(1, len(lines)):
    if lines[i] == "":
        stanzas.append(" ".join(lines[i_prev:i]))
        i_prev = i + 1

for z in stanzas[20:30]:
    print(z)

Za plecami Winstona głos z teleekranu nadal plótł coś o surówce i o tym, że wykonano, z nawiązką, dziewiąty Plan Trzyletni. Teleekran jednocześnie transmitował i odbierał dane. Rejestrował każdy wydany przez Winstona dźwięk, głośniejszy od najcichszego szeptu; co więcej, dopóki Winston przebywał w polu widzenia panelu, mógł być także obserwowany. Nikt oczywiście nie miał pewności, czy w danym momencie podlega obserwacji. Można było jedynie zgadywać, jak często i według jakiej logiki myślopolicja uruchamiała konkretny kanał przesyłu. Nie dało się wykluczyć, że obserwowali wszystkich przez cały czas. W każdym razie mogli podłączyć się do dowolnego kanału, kiedy tylko chcieli. Trzeba było żyć — i ludzie tak żyli, z przyzwyczajenia, które przeradzało się w instynkt — zakładając, że każdy dźwięk jest podsłuchiwany, a każdy ruch podglądany, z wyjątkiem chwil, gdy w pomieszczeniu jest ciemno.
Winston stał tyłem do teleekranu. Tak było bezpieczniej; chociaż, jak dobrze wiedział, z ludzkich ple

In [32]:
# stats
print(max(len(z) for z in stanzas))
print(sum(len(z) for z in stanzas)/len(stanzas))
words = set()
for z in stanzas:
    for w in z.split():
        words.add(w)
print(len(words))

3054
370.3955174686882
26775


In [ ]:
# prepare data

In [33]:
# add START/END symbols
x_txt = ["START " + z + " END" for z in stanzas]

In [35]:
# torchtext is deprecated
import keras

In [36]:
tv = keras.layers.TextVectorization(output_sequence_length=300)
tv.adapt(x_txt)

In [37]:
vocab = tv.get_vocabulary()
print(len(vocab))
vocab[:10]

19821


['',
 '[UNK]',
 np.str_('się'),
 np.str_('w'),
 np.str_('nie'),
 np.str_('—'),
 np.str_('i'),
 np.str_('start'),
 np.str_('end'),
 np.str_('na')]

In [38]:
# inverse vocab:
iw = dict()
for i, w in enumerate(vocab):
    iw[w] = i

In [39]:
xp = tv(x_txt)
xp[:10]

<tf.Tensor: shape=(10, 300), dtype=int64, numpy=
array([[    7, 17997, 14854, ...,     0,     0,     0],
       [    7,   502,   788, ...,     0,     0,     0],
       [    7, 17577, 19783, ...,     0,     0,     0],
       ...,
       [    7,  1321, 19227, ...,     0,     0,     0],
       [    7,    92,  6714, ...,     0,     0,     0],
       [    7,     8,     0, ...,     0,     0,     0]])>

In [40]:
xp = torch.tensor(xp.numpy())

In [41]:
y = xp[:,1:] # shift left: predict next word
x = xp[:,:-1]

In [42]:
print(x[0][:10])
print(y[0][:10])

tensor([    7, 17997, 14854,     8,     0,     0,     0,     0,     0,     0])
tensor([17997, 14854,     8,     0,     0,     0,     0,     0,     0,     0])


In [43]:
from torch.utils.data import DataLoader, TensorDataset

In [44]:
d_train = TensorDataset(torch.tensor(x), torch.tensor(y))
dl_train = DataLoader(d_train, batch_size=16, shuffle=True)

<ipython-input-44-62a7cd076784>:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  d_train = TensorDataset(torch.tensor(x), torch.tensor(y))


### Transformer elements

In [45]:
import numpy as np

In [46]:
def positional_encoding(length, depth):
  depth = depth/2
  positions = np.arange(length)[:, np.newaxis]
  depths = np.arange(depth)[np.newaxis, :]/depth
  pos_enc_complex = np.exp(1j*positions/(10000**depths))
  pos_enc = pos_enc_complex.view(np.float64)
  return pos_enc

In [47]:
class PositionBlock(nn.Module):
    def __init__(self, embed_dim, seq_length):
        super().__init__()
        pe = positional_encoding(length=seq_length, depth=embed_dim)
        pe = pe.astype(np.float32)[np.newaxis,...]
        self.pe = torch.tensor(pe, requires_grad=False).to(device)
    def forward(self, x):
        return x + self.pe

In [48]:
class TransformerBlock(nn.Module):
    def __init__(self, num_heads, embed_dim, dropout_rate=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.dropout_rate = dropout_rate
        self.mha = nn.MultiheadAttention(num_heads=num_heads,
                                         embed_dim=embed_dim,
                                         batch_first=True)
        self.FF = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
        )
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.dp = nn.Dropout(self.dropout_rate)

    def forward(self, inputs, mask):
        a, _ = self.mha(query=inputs, value=inputs, key=inputs,
                        need_weights=False,
                        is_causal=True,key_padding_mask=mask,
                        attn_mask=(nn.Transformer.generate_square_subsequent_mask(299, device=device) < -100))
        e2 = self.ln1(a + inputs)
        # feed-forward part
        e3 = self.FF(e2)
        e4 = self.ln2(e3 + e2)
        e4 = self.dp(e4)
        return e4

In [52]:
class LLM(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.d = d
        self.emb = nn.Embedding(len(vocab), d)
        self.pos = PositionBlock(d, 299)
        self.t1 = TransformerBlock(2, d)
        self.t2 = TransformerBlock(2, d)
        self.t3 = TransformerBlock(2, d)
        self.t4 = TransformerBlock(2, d)
        self.t5 = TransformerBlock(2, d)
        self.fl = nn.Flatten()
        self.l1 = nn.LazyLinear(len(vocab))
    def forward(self, x):
        mask = (x == 0).to(device)
        #mask = torch.where(mask, -torch.inf, 0)
        emb = self.emb(x)
        pos = self.pos(emb)
        t1 = self.t1(pos, mask)
        t2 = self.t2(t1, mask)
        t3 = self.t3(t2, mask)
        t4 = self.t4(t3, mask)
        t5 = self.t5(t4, mask)
        #x = self.fl(t2)
        x = self.l1(t2)
        return x

In [53]:
# training loop
import torch.optim as optim

def fit(net, train=dl_train, epochs=10,
        learning_rate=None):
    net = net.to(device)

    if learning_rate is None:
        optimizer = optim.Adam(net.parameters())
    else:
        optimizer = optim.Adam(net.parameters(), lr=learning_rate)
    #loss = torch.nn.CrossEntropyLoss()
    loss = torch.nn.CrossEntropyLoss(ignore_index=0)  # ignore loss on padding
    for e in range(epochs):
        net.train()
        epoch_loss = 0
        for X, y in tqdm(train):
            optimizer.zero_grad()
            X = X.to(device)
            y = y.to(device)
            pred = net(X)
            l = loss(pred.view(-1, len(vocab)), y.view(-1))
            epoch_loss += l.item()
            l.backward()
            optimizer.step()
        print("Epoch", e, "loss:", epoch_loss/len(train))

In [54]:
llm = LLM()
fit(llm)

  0%|          | 0/95 [00:00<?, ?it/s]

Epoch 0 loss: 8.704846372102436


  0%|          | 0/95 [00:00<?, ?it/s]

Epoch 1 loss: 7.904008654544228


  0%|          | 0/95 [00:00<?, ?it/s]

...

Epoch 7 loss: 6.557036158913061


  0%|          | 0/95 [00:00<?, ?it/s]

Epoch 8 loss: 6.29006747697529


  0%|          | 0/95 [00:00<?, ?it/s]

Epoch 9 loss: 6.01144061339529


In [55]:
fit(llm, epochs=10)

  0%|          | 0/95 [00:00<?, ?it/s]

Epoch 0 loss: 6.021484008588289


  0%|          | 0/95 [00:00<?, ?it/s]

Epoch 1 loss: 5.7955223183882865


  0%|          | 0/95 [00:00<?, ?it/s]

...

Epoch 7 loss: 4.6058958254362405


  0%|          | 0/95 [00:00<?, ?it/s]

Epoch 8 loss: 4.443945779298481


  0%|          | 0/95 [00:00<?, ?it/s]

Epoch 9 loss: 4.287414636110005


In [56]:
fit(llm, epochs=100)

  0%|          | 0/95 [00:00<?, ?it/s]

Epoch 0 loss: 4.496278562043843


  0%|          | 0/95 [00:00<?, ?it/s]

Epoch 1 loss: 4.328371572494507


  0%|          | 0/95 [00:00<?, ?it/s]

...

Epoch 97 loss: 1.146450286162527


  0%|          | 0/95 [00:00<?, ?it/s]

Epoch 98 loss: 1.133078167940441


  0%|          | 0/95 [00:00<?, ?it/s]

Epoch 99 loss: 1.1252867880620454


In [57]:
fit(llm, epochs=100, learning_rate=0.0001)

  0%|          | 0/95 [00:00<?, ?it/s]

Epoch 0 loss: 1.0478139896141856


  0%|          | 0/95 [00:00<?, ?it/s]

Epoch 1 loss: 1.0221384920571979


  0%|          | 0/95 [00:00<?, ?it/s]

...

Epoch 97 loss: 0.8439513884092632


  0%|          | 0/95 [00:00<?, ?it/s]

Epoch 98 loss: 0.8345356426740947


  0%|          | 0/95 [00:00<?, ?it/s]

Epoch 99 loss: 0.8483709435713919


In [62]:
x_t = torch.zeros(x.shape[1], dtype=int, device=device)
x_t[0] = iw["start"];t0=0 # start
x_t[1] = iw["obywatel"];t0=1
# x_t[2] = iw["cenzurowała"];t0=2
# x_t[3] = iw["wolność"];t0=3
llm.eval()
for t in range(t0, 30):
    #print(llm(x_t.view(1,-1)).argmax(axis=2).shape)
    x_t[t+1] = llm(x_t.view(1,-1)).argmax(axis=2)[0,t]
    for w in x_t:
        print(vocab[w], end=" ")
    print()

start obywatel tak                                                                                                                                                                                                                                                                                                         
start obywatel tak niczego                                                                                                                                                                                                                                                                                                        
start obywatel tak niczego to                                                                                                                                                                                                                                                                                                       
start obywatel tak niczego to wydani

In [60]:
x_t = torch.zeros(x.shape[1], dtype=int, device=device)
x_t[0] = iw["start"];t0=0 # start
x_t[1] = iw["winston"];t0=1
x_t[2] = iw["zamarł"];t0=2
x_t[3] = iw["gdy"];t0=3
x_t[4] = iw["zobaczył"];t0=4
#x_t[2] = iw["i"];t0=2
#x_t[3] = iw["wojski"];t0=3
llm.eval()
for t in range(t0, 20):
    logits = llm(x_t.view(1,-1))[0][t]
    logits /= 0.9 # temperature
    probs = F.softmax(logits, dim=-1)
    probs = probs.detach().cpu().numpy().astype(float)
    print(probs.sum())
    probs = probs/probs.sum()
    t_pred = np.flatnonzero(np.random.multinomial(1, probs)).item()
    x_t[t+1] = t_pred
    for w in x_t:
        print(vocab[w], end=" ")
    print()

1.0000000258816222
start winston zamarł gdy zobaczył na                                                                                                                                                                                                                                                                                                      
1.000000047527439
start winston zamarł gdy zobaczył na podłogę                                                                                                                                                                                                                                                                                                     
1.0000001049318687
start winston zamarł gdy zobaczył na podłogę opadł                                                                                                                                                                                                                                 